# Part 3: FinBERT Fine-Tuning (Local Execution)

This notebook covers:
- Loading pre-trained FinBERT model weights and tokenizer (`yiyanghkust/finbert-tone`)
- Building PyTorch DataLoaders from cached Financial PhraseBank splits
- Fine-tuning loop with AdamW, learning rate scheduling, and early stopping
- Evaluating classification metrics: Accuracy, Macro F1, Confusion Matrix

In [1]:
import sys
sys.path.append("..")
from config.global_config import CONFIG
from src.finbert_finetune import load_finbert_model, tokenize_dataset, train_finbert, evaluate_finbert
import pandas as pd

model, tokenizer = load_finbert_model(CONFIG["finbert_base_checkpoint"])
train_df = pd.read_parquet(f"{CONFIG['processed_dir']}/train.parquet")
val_df = pd.read_parquet(f"{CONFIG['processed_dir']}/val.parquet")
train_ds = tokenize_dataset(train_df["sentence"].tolist(), tokenizer, train_df["label"].tolist(), CONFIG["max_seq_len"])
val_ds = tokenize_dataset(val_df["sentence"].tolist(), tokenizer, val_df["label"].tolist(), CONFIG["max_seq_len"])
history = train_finbert(model, tokenizer, train_ds, val_ds, CONFIG["checkpoint_dir"],
    epochs=CONFIG["num_train_epochs"], batch_size=CONFIG["train_batch_size"],
    learning_rate=CONFIG["learning_rate"], seed=CONFIG["seed"])

C:\Users\Samarth\OneDrive\Documents\Programing\research paper code\retail-sentiment-micro-agents\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Samarth\OneDrive\Documents\Programing\research paper code\retail-sentiment-micro-agents\notebooks\..\src\finbert_finetune.py:161: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.943500,0.472970,0.829436,0.795367
2,0.314300,0.464526,0.840440,0.814000
3,0.135600,0.593255,0.843191,0.818362
4,0.050100,0.742194,0.843191,0.813924
5,0.018900,0.898437,0.833563,0.815776
6,0.007900,0.892301,0.840440,0.817828


In [2]:
test_df = pd.read_parquet(f"{CONFIG['processed_dir']}/test.parquet")
test_ds = tokenize_dataset(test_df["sentence"].tolist(), tokenizer, test_df["label"].tolist(), CONFIG["max_seq_len"])
test_metrics = evaluate_finbert(model, test_ds, batch_size=CONFIG["train_batch_size"])
print(test_metrics)

{'accuracy': 0.8528198074277854, 'macro_f1': 0.8329816526171153, 'f1_negative': 0.8177339901477833, 'f1_neutral': 0.8883720930232558, 'f1_positive': 0.7928388746803069}
